# PATHS: second independent training seed
Run one experiment at a time on a Colab GPU. This notebook uses the completed **hard24 v4** or **cr3cov v2** archive, not the older Run_1 archive.

It keeps the cohort, split, validation-prefix seed 145, evaluation patients, model hyperparameters, seven samplers, M=10,000, and rollout seeds 41001/52001/63001/74001/85001 fixed. It trains every neural model with seed 43. The two experiments use separate subprocesses to prevent identically named Python modules from colliding. No manuscript files are read or changed.

The code bundle contains no results. Keep the previous results ZIP available on Drive or upload it when prompted. The full runs have **not** been executed in preparing this notebook.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, json
from google.colab import drive, files
drive.mount('/content/drive')
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
                       'lifelines==0.30.3', 'pandas', 'scipy', 'matplotlib'])
CODE = Path('/content/PATHS_second_seed')
if not (CODE/'second_seed.py').is_file():
    print('Upload PATHS_second_seed_bundle.zip (the new CODE bundle).')
    uploaded=files.upload()
    archives=[Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if len(archives)!=1: raise ValueError('Upload exactly one code ZIP')
    destination=Path('/content').resolve()
    with zipfile.ZipFile(archives[0]) as archive:
        if 'PATHS_second_seed/second_seed.py' not in archive.namelist():
            raise ValueError('This is not PATHS_second_seed_bundle.zip')
        if any(not (destination/n).resolve().is_relative_to(destination) for n in archive.namelist()):
            raise ValueError('Invalid archive path')
        archive.extractall(destination)
print('Code:',CODE)


Choose `hard24` first, then repeat from this cell with `cr3`. If the suggested previous-run path is different on your Drive, change it below. If it is absent, the next cell offers to upload the correct results ZIP and finds its result root. Do not select the older Run_1_results archive for hard24.


In [ ]:
EXPERIMENT = 'hard24'  # 'hard24' or 'cr3'
TRAINING_SEED = 43
paths={
    'hard24': '/content/drive/MyDrive/Glazier_results/full_hard24_add_history_v4',
    'cr3': '/content/drive/MyDrive/Glazier_results/full_cr3cov_v2',
}
PREVIOUS_RUN=Path(paths[EXPERIMENT])  # replace with your actual extracted result root
OUT=Path('/content/drive/MyDrive/PATHS_results')/f'{EXPERIMENT}_training_seed_{TRAINING_SEED}'
if not (PREVIOUS_RUN/'training_settings.json').is_file():
    print('Upload the completed full_hard24_add_history_v4 results ZIP.' if EXPERIMENT=='hard24'
          else 'Upload the completed full_cr3cov_v2 results ZIP.')
    uploaded=files.upload()
    archives=[Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if len(archives)!=1: raise ValueError('Upload exactly one results ZIP')
    destination=(Path('/content/drive/MyDrive/PATHS_archives')/EXPERIMENT).resolve()
    destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        if any(not (destination/n).resolve().is_relative_to(destination) for n in archive.namelist()):
            raise ValueError('Invalid archive path')
        # Existing files are kept; the runner verifies their hashes before use.
        for member in archive.infolist():
            target=destination/member.filename
            if not target.exists(): archive.extract(member,destination)
    matches=[]
    for config in destination.rglob('training_settings.json'):
        candidate=config.parent
        if (candidate/'evaluation/evaluation_settings.json').is_file() and (candidate/'checkpoints/DynamicDeepHit.pt').is_file():
            matches.append(candidate)
    if len(matches)!=1:
        raise ValueError('Set PREVIOUS_RUN to one result root manually. Found: '+str(matches))
    PREVIOUS_RUN=matches[0]
assert (PREVIOUS_RUN/'checkpoints/cohort.npz').is_file(), PREVIOUS_RUN
print('Previous results:',PREVIOUS_RUN)
print('New results:',OUT)
print('Training seed:',TRAINING_SEED,'; fixed rollout seeds: 41001, 52001, 63001, 74001, 85001')


Preflight copies only the cohort/splits, deterministic references and true-generator oracle samples. It copies no trained neural model or learned prediction. It loads the actual archived architecture and loss settings, rather than the older notebook defaults. The preflight must succeed before training.

For the closest comparison use the same runtime as the saved runs: PyTorch 2.11.0+cu128 and NumPy 2.1.3; the CR archive reports an NVIDIA L4. Colab hardware/package changes can prevent bitwise repeatability even when the seed is fixed. The runner records the new environment. Do not change loss functions, hyperparameters or deterministic-kernel settings for just this replicate.


In [ ]:
environment=dict(os.environ)
environment['PYTHONHASHSEED']=str(TRAINING_SEED)  # takes effect in the fresh subprocess
environment['MKL_THREADING_LAYER']='SEQUENTIAL'
command=[sys.executable,'-u',str(CODE/'second_seed.py'),
         '--experiment',EXPERIMENT,'--previous',str(PREVIOUS_RUN),
         '--out',str(OUT),'--training-seed',str(TRAINING_SEED),'--device','cuda']
subprocess.run(command+['--prepare-only'],check=True,env=environment)
display(json.loads((OUT/'second_seed_plan.json').read_text()))


The next cell starts the full run. It trains all neural models and then evaluates all seven samplers at the original budget and seeds. It also exports ATLAS-labeled figures. Keep the runtime connected. Rerunning this cell resumes from completed model checkpoints/prediction files; an interrupted model is trained again from its seed, because optimizer states are not saved. Source/settings changes require a new output folder.


In [ ]:
log=OUT/'full_run.log'
with log.open('a',encoding='utf-8') as handle:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,env=environment)
    for line in process.stdout:
        print(line,end='');handle.write(line);handle.flush()
    returncode=process.wait()
if returncode: raise subprocess.CalledProcessError(returncode,command)
assert (OUT/'evaluation/COMPLETE.json').is_file()
print('Complete:',OUT)


Aggregate this experiment's two training seeds only after the new evaluation is complete. `per_checkpoint_MAE.csv` keeps each model fit separate; `two_training_seed_MAE.csv` reports the SD between the two checkpoint means separately from within-checkpoint rollout SD. The oracle and deterministic CPH are separated from stochastic neural training replication. Ten rollout results are **not ten trained models**.


In [ ]:
AGGREGATED=OUT.parent/f'{EXPERIMENT}_training_seeds_42_{TRAINING_SEED}'
subprocess.run([sys.executable,str(CODE/'aggregate_seeds.py'),
                '--first',str(PREVIOUS_RUN),'--second',str(OUT),
                '--destination',str(AGGREGATED)],check=True,env=environment)
import pandas as pd
display(pd.read_csv(AGGREGATED/'two_training_seed_MAE.csv'))
print('ATLAS figures:',OUT/'ATLAS_figures')


Optional: regenerate the seed-42 figures with the same ATLAS naming, without retraining. Original checkpoint names and model IDs intentionally remain unchanged. The new `display_name` column supplies names for plots/tables. For future custom plots use `atlas_figures.display_name(model, method)` for every tick label and legend.


In [ ]:
OLD_FIGURES=OUT.parent/f'{EXPERIMENT}_training_seed_42_ATLAS_figures'
subprocess.run([sys.executable,str(CODE/'atlas_figures.py'),
                '--experiment',EXPERIMENT,'--evaluation',str(PREVIOUS_RUN/'evaluation'),
                '--destination',str(OLD_FIGURES)],check=True,env=environment)
print('Seed-42 figures:',OLD_FIGURES)


After both experiments finish, the optional script `publication_views.py` can regenerate the two-panel survival-MAE overview and the individual baseline-patient figure from any matching pair of completed runs. For example:

```python
subprocess.run([sys.executable, str(CODE/'publication_views.py'),
    '--hard-run', '/content/drive/MyDrive/PATHS_results/hard24_training_seed_43',
    '--cr-run', '/content/drive/MyDrive/PATHS_results/cr3_training_seed_43',
    '--destination', '/content/drive/MyDrive/PATHS_results/ATLAS_overview_seed_43'], check=True)
```

Each figure is saved as PNG and SVG. No manuscript project is changed.
